# T13 - Web Scraping Test

**Course section:** 13 - Web Scraping
**Course notebooks covered:** `13-Web-Scraping/00-Guide-to-Web-Scraping.ipynb` and `13-Web-Scraping/01-Web-Scraping-Exercises.ipynb`

**14 questions / 20 points** (4 multiple choice at 1 point, 4 conceptual at 1 point, 6 coding exercises at 2 points).

**Suggested time budget:** 60 to 75 minutes.

**How the check cells work:** every coding exercise is followed by a cell tagged `check` that runs `assert` statements against the exact variable names your question asks for. Run those cells yourself before you trust your answer, and make sure you run the cells in order.

**Before you start:** work through `S0 - Testing Syllabus.ipynb` in the same folder for the overall plan and rules. Never put credentials in a notebook, and never write files into the course repository - use `tempfile` if a file is needed.

### Q1 - Multiple choice (1 point)

When you run `res = requests.get("http://www.example.com")`, what does `res` hold?

A. The raw HTML of the page as a Python `str`
B. A `requests.models.Response` object that holds the status code, headers and body
C. A BeautifulSoup object
D. A Python dictionary with keys "status", "headers" and "text"

<details>
<summary>Solution</summary>

**Correct answer: B**

`requests.get()` returns a `requests.models.Response` object. That object carries the HTTP status code (`res.status_code`), the response headers (`res.headers`) and the body, which you can reach as decoded text (`res.text`) or as raw bytes (`res.content`).

**Why:** the course creates the soup from `res.text`, never from `res` directly - `res` is the container that holds the page.

</details>

### Q2 - Multiple choice (1 point)

You downloaded an image and want to save the raw bytes to disk. Which attribute gives you exactly those bytes?

A. `res.text`
B. `res.content`
C. `res.headers`
D. `res.encoding`

<details>
<summary>Solution</summary>

**Correct answer: B**

`res.content` returns the raw bytes of the response body. `res.text` returns the same body decoded into a `str`, which is what you want for HTML, not for binary data such as a JPEG.

**Why:** in the course the image is written with `f.write(image_link.content)` and the file is opened in `'wb'` (binary write) mode, because `content` gives bytes.

</details>

### Q3 - Multiple choice (1 point)

You want every element on the page that has the CSS class `author`. Which call returns them?

A. `soup.select('author')`
B. `soup.select('.author')`
C. `soup.select('#author')`
D. `soup.select('class=author')`

<details>
<summary>Solution</summary>

**Correct answer: B**

In CSS selector syntax a class is written with a leading dot and an id with a leading `#`. So `soup.select('.author')` returns a list of every element carrying the class `author`, while `soup.select('#author')` would look for the id `author`.

**Why:** the scraping exercises use `soup.select(".author")` and then read `name.text` for each match.

</details>

### Q4 - Multiple choice (1 point)

Which line correctly saves a downloaded JPEG to disk?

A. `open('pic.jpg', 'w').write(res.content)`
B. `open('pic.jpg', 'wb').write(res.content)`
C. `open('pic.jpg', 'rb').write(res.content)`
D. `open('pic.jpg', 'w').write(res.text)`

<details>
<summary>Solution</summary>

**Correct answer: B**

`res.content` is bytes, so the file must be opened in binary write mode `'wb'`. Mode `'w'` is text mode and would raise a `TypeError` when handed bytes, `'rb'` is for reading, and `res.text` is a decoded string that would corrupt a binary file.

**Why:** the course explicitly notes "not the 'wb' call to denote a binary writing of the file" when saving the Deep Blue image.

</details>

### Q5 - Predict the output (1 point)

Assume `requests` and `bs4` are imported and the following code runs:

```text
res = requests.models.Response()
res.status_code = 200
res.encoding = 'utf-8'
res._content = b'<html><body><p>Hello</p></body></html>'
soup = bs4.BeautifulSoup(res.text, 'html.parser')
print(res.ok)
print(soup.find('p').text)
print(len(soup.select('p')))
```

What exactly does the script print? Write out the three lines of output before you open the solution block below.

<details>
<summary>Solution</summary>

**Correct output:**

```
True
Hello
1
```

**Why:** `res.ok` is `True` whenever the status code is below 400, so a status code of 200 means the request succeeded. `soup.find('p')` returns the *first* matching tag, and `.text` gives the text inside it (`Hello`). `soup.select('p')` returns a *list* of every matching tag, and there is exactly one paragraph, so `len(...)` is `1`.

Note that `soup.find` returns a single tag while `soup.select` and `soup.find_all` return lists.

</details>

### Q6 - Predict the output (1 point)

Assume `bs4` is imported and the following code runs:

```text
html = '<div><a href="/page/1">First</a><a>No link</a></div>'
soup = bs4.BeautifulSoup(html, 'html.parser')
print(soup.find('a').get('href'))
print(soup.find_all('a')[1].get('href', 'default'))
print(soup.select('a')[0].get_text())
```

What exactly does the script print? Write out the three lines of output before you open the solution block below.

<details>
<summary>Solution</summary>

**Correct output:**

```
/page/1
default
First
```

**Why:** `.get('href')` returns the value of the attribute when it exists. On the second anchor there is no `href` attribute at all, but `.get('href', 'default')` supplies the fallback `default` instead of raising. `.get_text()` returns the visible text of the tag.

**Why it matters:** the course reads image and link locations with `computer['src']` and `book.select('a')[1]['title']`, where a missing key would raise a `KeyError` - `.get()` is the safe version of the same idea.

</details>

### Q7 - Conceptual (1 point)

In the course, after scraping `src` from an image tag, the notebook adds `https://` in front of the link before calling `requests.get()` on it. Explain in your own words why that prefix is needed, and what would happen if you left it out.

<details>
<summary>Solution</summary>

**Correct answer:** the `src` value stored on the page is often a *relative* link (for example `upload.wikimedia.org/...` or `/images/pic.jpg`), not a complete URL. `requests.get()` needs a full URL including the scheme, so the scheme `https://` has to be put back in front of it.

**Why:** without the scheme `requests` cannot work out which server to talk to and raises a clear `MissingSchema` error - the course notes that the error message is "pretty descriptive" and that the fix is to add `https://`.

</details>

### Q8 - Conceptual (1 point)

The course lists three rules to follow when scraping. Name all three and add one sentence explaining why they matter for a long-running scraping job.

<details>
<summary>Solution</summary>

**Correct answer:** the three rules are

1. Always be respectful and get permission to scrape - do not bombard a website with requests, or your IP address may be blocked.
2. Be aware that websites change often, so your code can go from working to totally broken from one day to the next.
3. Almost every web scraping project of interest is a unique and custom job, so generalise the skills you learned.

**Why:** a scraper usually runs many times over weeks or months. Permission and a slow request rate keep you from being blocked, and the fact that pages change means you must expect to maintain and update your code instead of writing it once.

</details>

### Q9 - Coding exercise (2 points)

Build a response object **without any network access** and pull a title out of it.

Using `requests` and `bs4` (import them yourself), create a response object named **`res`** with

- `res.status_code` equal to `200`
- `res.encoding` equal to `'utf-8'`
- `res._content` set to the bytes of this HTML:

```html
<html><head><title>Example Domain</title></head><body><p>Hello</p></body></html>
```

Then create a soup object named **`soup`** from `res.text` using `bs4.BeautifulSoup` with the `'html.parser'` parser, and store the text of the `<title>` tag in a variable named **`title_text`**.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
import requests
import bs4

res = requests.models.Response()
res.status_code = 200
res.encoding = 'utf-8'
res._content = b'<html><head><title>Example Domain</title></head><body><p>Hello</p></body></html>'

soup = bs4.BeautifulSoup(res.text, 'html.parser')
title_text = soup.select('title')[0].get_text()
```

**Why:** Building the Response by hand is the offline equivalent of what `requests.get()` gives you, and `soup.select('title')[0].get_text()` is the same two-step the course uses for the page title.

</details>

In [ ]:
assert type(res).__name__ == 'Response'
assert res.status_code == 200
assert res.ok is True
assert 'Example Domain' in res.text
assert title_text == 'Example Domain'

### Q10 - Coding exercise (2 points)

Parse a page you already have in a string - no network needed.

Store this HTML in a variable named **`html`**:

```html
<html><body>
  <p>First paragraph</p>
  <p>Second paragraph</p>
  <p>Third paragraph</p>
</body></html>
```

Create a soup object named **`soup`** from `html` with `bs4.BeautifulSoup` and the `'html.parser'` parser, then store the text of every paragraph in a list named **`paragraphs`** (use `soup.find_all('p')` or `soup.select('p')` and the `.text` attribute of each tag).

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
import bs4

html = '<html><body>\n  <p>First paragraph</p>\n  <p>Second paragraph</p>\n  <p>Third paragraph</p>\n</body></html>'

soup = bs4.BeautifulSoup(html, 'html.parser')
paragraphs = [tag.text for tag in soup.find_all('p')]
```

**Why:** `find_all` gives every matching tag as a list, and `.text` pulls the visible text out of each one - the same pattern the course uses for quotes and authors.

</details>

In [ ]:
assert len(paragraphs) == 3
assert paragraphs[0] == 'First paragraph'
assert paragraphs[2] == 'Third paragraph'

### Q11 - Coding exercise (2 points)

Use a CSS class selector.

Store this HTML in a variable named **`html`**:

```html
<html><body>
  <div class="quote"><span class="text">To be or not to be</span><small class="author">Shakespeare</small></div>
  <div class="quote"><span class="text">I have a dream</span><small class="author">King</small></div>
  <div class="note"><small class="author">Anonymous</small></div>
</body></html>
```

Create a soup object named **`soup`** and use `soup.select('.author')` to store the author names in a list named **`authors`** (one name per element, using the `.text` attribute).

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
import bs4

html = '<html><body>\n  <div class="quote"><span class="text">To be or not to be</span><small class="author">Shakespeare</small></div>\n  <div class="quote"><span class="text">I have a dream</span><small class="author">King</small></div>\n  <div class="note"><small class="author">Anonymous</small></div>\n</body></html>'

soup = bs4.BeautifulSoup(html, 'html.parser')
authors = [tag.text for tag in soup.select('.author')]
```

**Why:** A leading dot means 'class', so `.author` picks up all three authors no matter which parent element they sit in - this is exactly the `.author` selector used on quotes.toscrape.com.

</details>

In [ ]:
assert authors == ['Shakespeare', 'King', 'Anonymous']
assert len(authors) == 3

### Q12 - Coding exercise (2 points)

Collect the links of a page.

Store this HTML in a variable named **`html`**:

```html
<html><body>
  <a href="/page/1">One</a>
  <a href="/page/2">Two</a>
  <a href="/page/3">Three</a>
</body></html>
```

Create a soup object named **`soup`** and store the `href` of every anchor in a list named **`links`**. Use the `.get()` method on each tag (for example `tag.get('href')`) so that the code does not crash on a tag without the attribute.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
import bs4

html = '<html><body>\n  <a href="/page/1">One</a>\n  <a href="/page/2">Two</a>\n  <a href="/page/3">Three</a>\n</body></html>'

soup = bs4.BeautifulSoup(html, 'html.parser')
links = [tag.get('href') for tag in soup.find_all('a')]
```

**Why:** `.get('href')` reads the attribute safely, which is the same accessor the course uses for `src` on image tags.

</details>

In [ ]:
assert links == ['/page/1', '/page/2', '/page/3']
assert all(link.startswith('/page/') for link in links)

### Q13 - Coding exercise (2 points)

Scrape image sources out of a page.

Store this HTML in a variable named **`html`**:

```html
<html><body>
  <img src="/static/img/one.jpg" alt="One">
  <img alt="No source">
  <img src="/static/img/two.jpg" alt="Two">
</body></html>
```

Create a soup object named **`soup`** and store the `src` of every image in a list named **`image_sources`**. Use `.get('src', '')` so that the image without a `src` contributes an empty string instead of raising an error.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
import bs4

html = '<html><body>\n  <img src="/static/img/one.jpg" alt="One">\n  <img alt="No source">\n  <img src="/static/img/two.jpg" alt="Two">\n</body></html>'

soup = bs4.BeautifulSoup(html, 'html.parser')
image_sources = [tag.get('src', '') for tag in soup.find_all('img')]
```

**Why:** The course reads `computer['src']` for the image it wants; `.get('src', '')` is the safe form and keeps the order of the images on the page.

</details>

In [ ]:
assert image_sources == ['/static/img/one.jpg', '', '/static/img/two.jpg']
assert len(image_sources) == 3

### Q14 - Coding exercise (2 points)

Work out the "parent" location of a URL.

Store the URL string `'http://books.toscrape.com/catalogue/page-1.html'` in a variable named **`url`** and store the URL pattern string `'http://books.toscrape.com/catalogue/page-{}.html'` in a variable named **`base_url`**.

Then create:

- **`page_url`** - the URL of page 3, built with `base_url.format(3)`
- **`parent_url`** - the URL of the parent catalogue directory, obtained from `url` with `urllib.parse.urljoin(url, '.')`

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
from urllib.parse import urljoin

url = 'http://books.toscrape.com/catalogue/page-1.html'
base_url = 'http://books.toscrape.com/catalogue/page-{}.html'

page_url = base_url.format(3)
parent_url = urljoin(url, '.')
```

**Why:** `base_url.format(n)` is the page loop from the course, and `urljoin(url, '.')` walks one directory up so you can crawl back to the parent listing page.

</details>

In [ ]:
assert page_url == 'http://books.toscrape.com/catalogue/page-3.html'
assert parent_url == 'http://books.toscrape.com/catalogue/'
assert url == 'http://books.toscrape.com/catalogue/page-1.html'

## Topics covered

- The request/response cycle: `requests.get`, the `requests.models.Response` object
- Status codes: `res.status_code`, `res.ok`, `raise_for_status`
- `response.text` versus `response.content`, and `response.headers`
- BeautifulSoup: `BeautifulSoup`, `soup.find`, `soup.find_all`, `soup.select`, `.text`, `.get`, `.get_text()`
- CSS selector syntax: tag names, `.class`, `#id`
- Crawling structure: page URL patterns with `.format()`, navigating to a parent page
- Scraping images: reading `src`, making a full URL, saving binary content with `'wb'`
- Respectful scraping: permission, request rate, and pages that change